# Telco Customer Churn — Modeling, Model Selection and Tableau Export

This notebook contains the final Machine Learning workflow:

1. Load the clean dataset.
2. Create train and test sets.
3. Preprocess numerical and categorical variables.
4. Train Logistic Regression, Decision Tree and Random Forest.
5. Tune tree-based models using cross-validation.
6. Compare the final models.
7. Apply SMOTE only to the training data.
8. Select the final business model.
9. Export customer-level predictions and summary tables for Tableau.

> The test dataset remains untouched until final evaluation.

## 0. Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TARGET = "churn_flag"

## 1. Load the clean dataset

The cell below searches a few common locations. Adjust `candidate_paths` only if your file is stored elsewhere.

In [ ]:
candidate_paths = [
    Path("telco_churn_clean.csv"),
    Path("../data/telco_churn_clean.csv"),
    Path("../data/clean/telco_churn_clean.csv"),
    Path("data/telco_churn_clean.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "telco_churn_clean.csv was not found. "
        "Place it beside this notebook or update candidate_paths."
    )

df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' is missing.")

print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", int(df.isna().sum().sum()))
print("\nTarget distribution:")
display(
    pd.DataFrame({
        "count": df[TARGET].value_counts().sort_index(),
        "percentage": (
            df[TARGET]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
        )
    })
)

## 2. Define features and create the train-test split

The split is stratified so that both datasets preserve approximately the same churn proportion.

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 3. Preprocessing

- Numerical variables are standardized using parameters learned from the training data.
- Categorical variables are one-hot encoded using categories learned from the training data.
- The customer index is preserved to support the Tableau export later.

In [ ]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

In [ ]:
scaler = StandardScaler()

X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train[numerical_features]),
    columns=numerical_features,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    scaler.transform(X_test[numerical_features]),
    columns=numerical_features,
    index=X_test.index
)

In [ ]:
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary",
        sparse_output=False
    )
except TypeError:
    # Compatibility with older scikit-learn versions
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary",
        sparse=False
    )

X_train_cat_array = encoder.fit_transform(X_train[categorical_features])
X_test_cat_array = encoder.transform(X_test[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat = pd.DataFrame(
    X_train_cat_array,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat_array,
    columns=encoded_feature_names,
    index=X_test.index
)

In [ ]:
X_train_processed = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

assert list(X_train_processed.columns) == list(X_test_processed.columns)
assert X_train_processed.isna().sum().sum() == 0
assert X_test_processed.isna().sum().sum() == 0

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

## 4. Reusable evaluation function

In [ ]:
def evaluate_classification_model(model, X_eval, y_eval, model_name):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    return pd.DataFrame({
        "Model": [model_name],
        "Accuracy": [accuracy_score(y_eval, y_pred)],
        "Precision": [precision_score(y_eval, y_pred, zero_division=0)],
        "Recall": [recall_score(y_eval, y_pred, zero_division=0)],
        "F1-score": [f1_score(y_eval, y_pred, zero_division=0)],
        "ROC-AUC": [roc_auc_score(y_eval, y_proba)]
    })


def overfitting_check(model, model_name):
    train_accuracy = model.score(X_train_processed, y_train)
    test_accuracy = model.score(X_test_processed, y_test)

    return pd.DataFrame({
        "Model": [model_name],
        "Training Accuracy": [train_accuracy],
        "Test Accuracy": [test_accuracy],
        "Accuracy Gap": [train_accuracy - test_accuracy]
    })

## 5. Logistic Regression baseline

In [ ]:
log_reg = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)

log_reg.fit(X_train_processed, y_train)

log_reg_results = evaluate_classification_model(
    log_reg,
    X_test_processed,
    y_test,
    "Logistic Regression"
)

log_reg_results.round(3)

## 6. Decision Tree baseline

In [ ]:
baseline_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

baseline_tree.fit(X_train_processed, y_train)

baseline_tree_results = evaluate_classification_model(
    baseline_tree,
    X_test_processed,
    y_test,
    "Decision Tree — Unrestricted"
)

baseline_tree_results.round(3)

In [ ]:
overfitting_check(
    baseline_tree,
    "Decision Tree — Unrestricted"
).round(3)

## 7. Decision Tree hyperparameter tuning

`GridSearchCV` selects the best depth using only the training data through five-fold cross-validation.

In [ ]:
tree_param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None]
}

tree_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    param_grid=tree_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

tree_grid_search.fit(
    X_train_processed,
    y_train
)

print("Best parameters:", tree_grid_search.best_params_)
print("Best cross-validation ROC-AUC:", round(tree_grid_search.best_score_, 3))

In [ ]:
tree_cv_results = pd.DataFrame(tree_grid_search.cv_results_)

tree_depth_results = (
    tree_cv_results[
        ["param_max_depth", "mean_test_score", "std_test_score"]
    ]
    .rename(columns={
        "param_max_depth": "max_depth",
        "mean_test_score": "mean_cv_roc_auc",
        "std_test_score": "std_cv_roc_auc"
    })
    .copy()
)

tree_depth_results["max_depth_label"] = tree_depth_results["max_depth"].apply(
    lambda value: "Unlimited" if pd.isna(value) else str(int(value))
)

tree_depth_results.round(3)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    tree_depth_results["max_depth_label"],
    tree_depth_results["mean_cv_roc_auc"],
    marker="o"
)

plt.title("Decision Tree Cross-Validation ROC-AUC by max_depth")
plt.xlabel("max_depth")
plt.ylabel("Mean CV ROC-AUC")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Final Decision Tree model

The best cross-validated estimator is evaluated once on the untouched test dataset.

In [ ]:
decision_tree = tree_grid_search.best_estimator_

decision_tree_results = evaluate_classification_model(
    decision_tree,
    X_test_processed,
    y_test,
    "Decision Tree Tuned"
)

decision_tree_results.round(3)

In [ ]:
tree_fit_comparison = pd.concat(
    [
        overfitting_check(
            baseline_tree,
            "Decision Tree — Unrestricted"
        ),
        overfitting_check(
            decision_tree,
            f"Decision Tree — max_depth={tree_grid_search.best_params_['max_depth']}"
        )
    ],
    ignore_index=True
)

tree_fit_comparison.round(3)

## 8. Random Forest baseline

In [ ]:
random_forest = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest.fit(X_train_processed, y_train)

random_forest_results = evaluate_classification_model(
    random_forest,
    X_test_processed,
    y_test,
    "Random Forest Baseline"
)

random_forest_results.round(3)

In [ ]:
overfitting_check(
    random_forest,
    "Random Forest Baseline"
).round(3)

## 9. Random Forest hyperparameter tuning

In [ ]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 8, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

rf_grid_search.fit(
    X_train_processed,
    y_train
)

print("Best parameters:", rf_grid_search.best_params_)
print("Best cross-validation ROC-AUC:", round(rf_grid_search.best_score_, 3))

In [ ]:
best_random_forest = rf_grid_search.best_estimator_

best_rf_results = evaluate_classification_model(
    best_random_forest,
    X_test_processed,
    y_test,
    "Random Forest Tuned"
)

best_rf_results.round(3)

In [ ]:
rf_fit_comparison = pd.concat(
    [
        overfitting_check(
            random_forest,
            "Random Forest Baseline"
        ),
        overfitting_check(
            best_random_forest,
            "Random Forest Tuned"
        )
    ],
    ignore_index=True
)

rf_fit_comparison.round(3)

## 10. Final model comparison

In [ ]:
final_model_comparison = pd.concat(
    [
        log_reg_results,
        decision_tree_results,
        best_rf_results
    ],
    ignore_index=True
)

final_model_comparison.round(3)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    final_model_comparison["Model"],
    final_model_comparison["ROC-AUC"]
)
plt.title("ROC-AUC Comparison")
plt.ylabel("ROC-AUC")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    final_model_comparison["Model"],
    final_model_comparison["Recall"]
)
plt.title("Recall Comparison")
plt.ylabel("Recall")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 11. Balance the training data with SMOTE

SMOTE is applied only to the processed training dataset. The original test dataset remains unchanged.

In [ ]:
original_distribution = pd.DataFrame({
    "Count": y_train.value_counts().sort_index(),
    "Percentage": (
        y_train.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

original_distribution.index = ["No Churn (0)", "Churn (1)"]
original_distribution

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

print("Original training shape:", X_train_processed.shape)
print("Balanced training shape:", X_train_smote.shape)

In [ ]:
balanced_distribution = pd.DataFrame({
    "Count": y_train_smote.value_counts().sort_index(),
    "Percentage": (
        y_train_smote.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

balanced_distribution.index = ["No Churn (0)", "Churn (1)"]
balanced_distribution

## 12. Retrain the selected Random Forest with SMOTE

In [ ]:
rf_smote = RandomForestClassifier(
    **rf_grid_search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_smote.fit(
    X_train_smote,
    y_train_smote
)

rf_smote_results = evaluate_classification_model(
    rf_smote,
    X_test_processed,
    y_test,
    "Random Forest + SMOTE"
)

rf_smote_results.round(3)

## 13. Compare before and after SMOTE

In [ ]:
smote_comparison = pd.concat(
    [
        best_rf_results,
        rf_smote_results
    ],
    ignore_index=True
)

smote_comparison.round(3)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    smote_comparison["Model"],
    smote_comparison["Recall"]
)
plt.title("Recall Before and After SMOTE")
plt.ylabel("Recall")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    smote_comparison["Model"],
    smote_comparison["Accuracy"]
)
plt.title("Accuracy Before and After SMOTE")
plt.ylabel("Accuracy")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 14. Select the final business model

For churn prevention, Recall is particularly important because a false negative represents a customer who is likely to leave but is not flagged for intervention.

The final choice can therefore favour `Random Forest + SMOTE` when the business cost of missing a churner is greater than the cost of contacting additional customers.

In [ ]:
final_model = rf_smote
final_model_name = "Random Forest + SMOTE"

print("Selected final model:", final_model_name)

## 15. Customer-level predictions for Tableau

The export contains:

- original customer attributes;
- actual churn outcome;
- predicted class;
- churn probability;
- prediction outcome category;
- whether the prediction was correct.

This table supports customer segmentation, risk-band analysis and confusion-matrix visualisations in Tableau.

In [ ]:
test_predictions = X_test.copy()

test_predictions["actual_churn"] = y_test
test_predictions["predicted_churn"] = final_model.predict(X_test_processed)
test_predictions["churn_probability"] = final_model.predict_proba(
    X_test_processed
)[:, 1]

test_predictions["prediction_correct"] = (
    test_predictions["actual_churn"]
    == test_predictions["predicted_churn"]
)

conditions = [
    (test_predictions["actual_churn"] == 1)
    & (test_predictions["predicted_churn"] == 1),

    (test_predictions["actual_churn"] == 0)
    & (test_predictions["predicted_churn"] == 0),

    (test_predictions["actual_churn"] == 0)
    & (test_predictions["predicted_churn"] == 1),

    (test_predictions["actual_churn"] == 1)
    & (test_predictions["predicted_churn"] == 0),
]

choices = [
    "True Positive",
    "True Negative",
    "False Positive",
    "False Negative",
]

test_predictions["prediction_outcome"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

test_predictions["risk_band"] = pd.cut(
    test_predictions["churn_probability"],
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=["Low", "Medium", "High", "Very High"],
    include_lowest=True
)

test_predictions.head()

## 16. Feature importance for Tableau

In [ ]:
feature_importance = (
    pd.DataFrame({
        "Feature": X_train_processed.columns,
        "Importance": final_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance["Rank"] = (
    feature_importance["Importance"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

feature_importance.head(15)

## 17. Export Tableau-ready CSV files

Three files are created in `tableau_exports/`:

1. `tableau_customer_predictions.csv`
2. `tableau_model_metrics.csv`
3. `tableau_feature_importance.csv`

In [ ]:
export_dir = Path("tableau_exports")
export_dir.mkdir(parents=True, exist_ok=True)

customer_export_path = (
    export_dir / "tableau_customer_predictions.csv"
)
metrics_export_path = (
    export_dir / "tableau_model_metrics.csv"
)
importance_export_path = (
    export_dir / "tableau_feature_importance.csv"
)

test_predictions.reset_index(
    names="source_row_id"
).to_csv(
    customer_export_path,
    index=False
)

all_model_metrics = pd.concat(
    [
        final_model_comparison,
        rf_smote_results
    ],
    ignore_index=True
)

all_model_metrics.to_csv(
    metrics_export_path,
    index=False
)

feature_importance.to_csv(
    importance_export_path,
    index=False
)

print("Created:")
print(customer_export_path.resolve())
print(metrics_export_path.resolve())
print(importance_export_path.resolve())

## 18. Suggested Tableau views

### Dashboard 1 — Model Performance
- KPI cards: Accuracy, Precision, Recall, F1-score and ROC-AUC.
- Side-by-side comparison of all evaluated models.
- Before-versus-after SMOTE comparison.
- Confusion matrix using `prediction_outcome`.

### Dashboard 2 — Customer Churn Risk
- Customers by `risk_band`.
- Average churn probability by contract type, tenure group, payment method or service type.
- Actual versus predicted churn by customer segment.
- Top features influencing the model.

### Business interpretation
The dashboard should clearly distinguish between:

- overall predictive performance;
- ability to detect churners;
- operational trade-off caused by SMOTE;
- customer groups that should be prioritised for retention actions.

## 19. Final checklist

Before delivery:

- Restart the kernel and run all cells.
- Confirm there are no red error cells.
- Verify that the three CSV files were created.
- Add `imbalanced-learn` to `requirements.txt`.
- Commit the notebook and Tableau export files to the appropriate project folders.
- Update the README with the final selected model and key results.